# Text Cleaning

## Load Dataset

In [1]:
import pandas as pd

In [2]:
headlines = pd.read_excel("headlines_Working.xlsx")

In [3]:
headlines.shape

(133375, 5)

In [4]:
headlines['RIC'].value_counts()

RIC
INF.L        4109
CEZP.PR      3305
AAPL.OQ      3024
JNJ.N        2152
COR          1888
             ... 
ITP.A           1
OBLG.OQ         1
VINCIT.HE       1
CPC.L           1
PW.A            1
Name: count, Length: 2221, dtype: int64

In [5]:
headlines = headlines.drop('Unnamed: 0', axis = 1)

In [6]:
headlines = headlines.dropna()

In [7]:
headlines.shape

(132063, 4)

In [8]:
from IPython.display import display
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', True) 

display(headlines.head)

<bound method NDFrame.head of             RIC                sdate                edate  \
0           COR  2021-10-16 00:00:00  2021-11-10 00:00:00   
1           COR  2021-10-16 00:00:00  2021-11-10 00:00:00   
2           COR  2021-10-16 00:00:00  2021-11-10 00:00:00   
3           COR  2021-10-16 00:00:00  2021-11-10 00:00:00   
4           COR  2021-10-16 00:00:00  2021-11-10 00:00:00   
...         ...                  ...                  ...   
133370  HYFM.OQ           1969-12-02           1969-12-27   
133371  HYFM.OQ           1969-12-02           1969-12-27   
133372  HYFM.OQ           1969-12-02           1969-12-27   
133373  HYFM.OQ           1969-12-02           1969-12-27   
133374  HYFM.OQ           1969-12-02           1969-12-27   

                                                                                                               Headlines  
0                                                    CIF/FOB Gulf Grain-Corn barge basis steady to lower, soybeans

In [9]:
df = headlines

## Text Formatting

## Clean Text 

In [10]:
!pip install langdetect

Defaulting to user installation because normal site-packages is not writeable


In [11]:
!wget -nc https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin

File ‘lid.176.bin’ already there; not retrieving.



In [12]:
from joblib import Parallel, delayed


In [13]:
# Step 2: Imports
import re
import fasttext
from joblib import Parallel, delayed

# Step 3: Load the fastText language model
ft_model = fasttext.load_model("lid.176.bin")

# Step 4: Fast regex-based prefilter
regex_platts = re.compile(r'platts:.*?(daily|report|assessment|commentary|rationale|summary)', re.IGNORECASE)
regex_dj = re.compile(r'dj cbot corn options close.*', re.IGNORECASE)

def is_potentially_valid(text):
    if not isinstance(text, str): return False
    if regex_platts.search(text): return False
    if regex_dj.search(text): return False
    return True

# Filter with fast regex logic
df = df[df['Headlines'].apply(is_potentially_valid)].reset_index(drop=True)
print(f"After regex filtering: {len(df)} rows")

# Step 5: Language detection using fasttext
def fast_lang_detect(text):
    if len(text.strip()) <= 5:
        return True  # Assume short lines are okay
    lang = ft_model.predict(text)[0][0]
    return lang == '__label__en'

# Parallelize language detection using joblib
lang_filter = Parallel(n_jobs=16, backend="threading")(
    delayed(fast_lang_detect)(text) for text in df['Headlines']
)

# Final filtered dataframe
df = df[lang_filter].reset_index(drop=True)
print(f"After language detection: {len(df)} rows")

After regex filtering: 131612 rows
After language detection: 129249 rows


In [14]:
df.shape

(129249, 4)

In [15]:
import re
from joblib import Parallel, delayed
import multiprocessing

# Precompile all regex patterns
regexes = [
    re.compile(r'\bpercent\b', re.IGNORECASE),  # "percent" → "%"
    re.compile(r'\bplatts\s*:\s*\d+\b', re.IGNORECASE),
    re.compile(r'(?<=\w)/(?=\w)'),  # Replace "/" between words
    re.compile(r'\b(19|20)\d{2}[-/](19|20)?\d{2}\b'),  # 2021-22
    re.compile(r'\b(?:NASDAQ|NYSE|AMEX):(\w{1,6})\b', re.IGNORECASE),  # Keep only ticker

    # ⬇️ NEW: Single super-regex for months/dates (combines many patterns cleanly)
    re.compile(r'\b(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)\s*[\-–/]?\s*\d{1,2}(?:st|nd|rd|th)?(?:[\-–/]\d{1,2})?(?:\s+\d{2,4})?\b', re.IGNORECASE),

    re.compile(r'\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b', re.IGNORECASE),
    re.compile(r'\b(dj|usda)\b', re.IGNORECASE),
    re.compile(r'\(\d+\s+pages\)', re.IGNORECASE),
    re.compile(r'<.*?>'),  # HTML tags
    re.compile(r'\b(19|20)\d{2}\b'),  # Years
    re.compile(r'[:,?]'),  # punctuation
    re.compile(r'\b\d{1,2}[–-]\d{1,2}\b'),  # standalone 1-2 day ranges
    re.compile(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b'),  # date formats like 01/01/2024
    re.compile(r'[\(\)\[\]\{\}\|]'),  # brackets
    re.compile(r'[–-]{1,2}')  # dashes
]

# Merge tokens like "$ 20" → "$20" and "15 %" → "15%"
def merge_symbols(text):
    tokens = text.split()
    merged = []
    i = 0
    while i < len(tokens):
        current = tokens[i]
        next_token = tokens[i + 1] if i + 1 < len(tokens) else ""

        if current == "$" and next_token.replace('.', '', 1).isdigit():
            merged.append(current + next_token)
            i += 2
        elif current.replace('.', '', 1).isdigit() and next_token == "%":
            merged.append(current + next_token)
            i += 2
        else:
            merged.append(current)
            i += 1
    return " ".join(merged)

# Main fast cleaner function
def cleaned_text_fast(text):
    if not isinstance(text, str):
        return text
    for i, pattern in enumerate(regexes):
        if i == 0:  # "percent" → "%"
            text = pattern.sub('%', text)
        elif i == 4:  # NASDAQ:HYFM → HYFM
            text = pattern.sub(r'\1', text)
        else:
            text = pattern.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return merge_symbols(text)

# Apply to all headlines using all CPU cores
cleaned = Parallel(n_jobs=multiprocessing.cpu_count())(
    delayed(cleaned_text_fast)(text) for text in df['Headlines']
)

# Store result
df['Cleaned_Headlines'] = cleaned

## Tokenization

In [16]:
!pip install spacy

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 68.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.2.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.5 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.5 which is incompatible.


In [17]:
!python -m spacy download en_core_web_sm

Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/spacy/__init__.py", line 6, in <module>
    from .errors import setup_default_warnings
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/spacy/errors.py", line 3, in <module>
    from .compat import Literal
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/spacy/compat.py", line 4, in <module>
    from thinc.util import copy_array
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/thinc/__init__.py", line 5, in <module>
    from .config import registry
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/thinc/config.py", line 5, in <module>
    from .types import Decorator
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/thinc/types.py", line 27, in <module>
    from .

In [18]:
import spacy
import pandas as pd

# Load spaCy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "tagger"])

# Merge function
def merge_symbols(tokens):
    merged = []
    i = 0
    while i < len(tokens):
        current = tokens[i]
        next_token = tokens[i + 1] if i + 1 < len(tokens) else ""

        if current == "$" and next_token.replace('.', '', 1).isdigit():
            merged.append(current + next_token)
            i += 2
        elif current.replace('.', '', 1).isdigit() and next_token == "%":
            merged.append(current + next_token)
            i += 2
        else:
            merged.append(current)
            i += 1
    return merged

# Tokenize headlines in batches (uses multiple threads internally)
texts = df["Cleaned_Headlines"].fillna("").astype(str).str.lower().tolist()

tokens = []
for doc in nlp.pipe(texts, batch_size=1000, n_process=1):  # Use n_process=1 to avoid multiprocessing errors
    raw_tokens = [token.text for token in doc if token.text.strip()]
    tokens.append(merge_symbols(raw_tokens))

df["Tokens"] = tokens

/jet/home/sadapa/.conda/envs/clean_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/jet/home/sadapa/.conda/envs/clean_env/lib/python3.10/site-packages/spacy/pipeline/lemmatizer.py:211: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


## Lemmatization

## NER

### NER using en_core_web_sm

In [19]:
import spacy

# Load spaCy with all pipeline components
nlp = spacy.load("en_core_web_sm")  # Keep full pipeline for NER

# Function to extract named entities from a spaCy doc
def get_ents(doc):
    return [(ent.text, ent.label_) for ent in doc.ents]

# Convert text column to list
texts = df["Cleaned_Headlines"].fillna("").astype(str).tolist()

# Use n_process > 1 to enable parallelism
ner_results = []
for doc in nlp.pipe(texts, batch_size=1000, n_process=4):  # Adjust n_process to your core count
    ner_results.append(get_ents(doc))

# Assign back to DataFrame
df["NER"] = ner_results

In [20]:
!python -m spacy download en_core_web_trf

Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/spacy/__init__.py", line 6, in <module>
    from .errors import setup_default_warnings
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/spacy/errors.py", line 3, in <module>
    from .compat import Literal
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/spacy/compat.py", line 4, in <module>
    from thinc.util import copy_array
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/thinc/__init__.py", line 5, in <module>
    from .config import registry
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/thinc/config.py", line 5, in <module>
    from .types import Decorator
  File "/jet/home/sadapa/.local/lib/python3.12/site-packages/thinc/types.py", line 27, in <module>
    from .

In [21]:
!pip install spacy[transformers]

Defaulting to user installation because normal site-packages is not writeable


### NER using en_core_web_trf

In [22]:
import spacy

# Load the transformer-based spaCy model
nlp = spacy.load("en_core_web_trf")  # This includes the transformer pipeline by default

# Function to extract NER from a spaCy Doc
def get_ents(doc):
    return [(ent.text, ent.label_) for ent in doc.ents]

# Get texts
texts = df["Cleaned_Headlines"].fillna("").astype(str).tolist()

# Use spaCy's efficient nlp.pipe for batching
ner_results = []
for doc in nlp.pipe(texts, batch_size=32):  # Transformer model: small batch size recommended (e.g., 8–32)
    ner_results.append(get_ents(doc))

df["NER2"] = ner_results

### Combining the results

In [23]:
from collections import defaultdict

def combine_ner_lists(ner1, ner2):
    combined = defaultdict(set)
    
    for text, label in ner1 + ner2:
        combined[(text.lower(), label)].add(text)  # case-insensitive match

    # Keep only one version of each entity (you can prefer trf text version if needed)
    return [(list(texts)[0], label) for (text_lower, label), texts in combined.items()]

# Combine row-wise
df["NER_combined"] = [
    combine_ner_lists(n1, n2)
    for n1, n2 in zip(df["NER"], df["NER2"])
]

# option 2: linking named entities to class labels (peer or target)

### Combining NERs per company

In [24]:
df.head()

,RIC,sdate,edate,Headlines,Cleaned_Headlines,Tokens,NER,NER2,NER_combined
0,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,"CIF/FOB Gulf Grain-Corn barge basis steady to lower, soybeans mixed",CIF FOB Gulf Grain Corn barge basis steady to lower soybeans mixed,"[cif, fob, gulf, grain, corn, barge, basis, steady, to, lower, soybeans, mixed]","[(CIF, ORG)]","[(CIF FOB, ORG)]","[(CIF, ORG), (CIF FOB, ORG)]"
1,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,"Export Summary-Philippine importer buys feed wheat, South Korea seeks soybeans",Export Summary Philippine importer buys feed wheat South Korea seeks soybeans,"[export, summary, philippine, importer, buys, feed, wheat, south, korea, seeks, soybeans]","[(Philippine, NORP), (South Korea, GPE)]","[(Philippine, NORP), (South Korea, GPE)]","[(Philippine, NORP), (South Korea, GPE)]"
2,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,DJ Thailand Corn Weather - Nov 9,Thailand Corn Weather,"[thailand, corn, weather]","[(Thailand, GPE)]","[(Thailand, GPE)]","[(Thailand, GPE)]"
3,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,DJ Northeast China Corn Weather - Nov 9,Northeast China Corn Weather,"[northeast, china, corn, weather]","[(Northeast, ORG), (China, GPE)]","[(China, GPE)]","[(Northeast, ORG), (China, GPE)]"
4,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,Corn: Cargo market without buyers,Corn Cargo market without buyers,"[corn, cargo, market, without, buyers]",[],[],[]


In [25]:
# Assumes df already has 'RIC' and 'Normalized_Entities' columns
grouped_df = df.groupby("RIC")["NER_combined"].apply(lambda x: sum(x, [])).reset_index()
grouped_df.columns = ["RIC", "NER_combined"]

In [26]:
display(grouped_df.head(10))

,RIC,NER_combined
0,000150.KS,"[(Second Quarter, DATE), (11 cent, MONEY), (CERV Cervus Equipment, ORG), (Cervus Equipment Corporation Cervus, ORG), (Second Quarter, DATE), (Cervus Equipment Corporation, ORG), (Cervus, ORG), (Press Release Cervus, ORG), (Second Quarter, DATE), (Cervus, ORG), (Second Quarter, DATE), (Cervus, ORG), (Asia Pacific Corporate, ORG), (13, CARDINAL), (Asia Pacific Corporate Calendar, EVENT), (Month Ahead, DATE), (13, DATE), (Doosan, PRODUCT), (Korean, NORP), (06 09, CARDINAL), (NH Investment & Securities Previously Woori Research, ORG), (Second Half, DATE), (06 09, DATE), (MSCI ESG Research, ORG), (Doosan Corporation, ORG), (Korea, GPE), (SK, ORG), (Cervus Equipment Corporation Cervus Equipment Peterbilt, ORG), (Open New TRP Parts Location, ORG), (Cervus Equipment Corporation, ORG), (Cervus Equipment Peterbilt, ORG), (Prince Albert, GPE), (Saskatchewan, GPE), (200, CARDINAL), (6, CARDINAL), (REG Ceres Power Holdings Notice of Capital Markets Teach, ORG), (REG Ceres Power Holdings, ORG), (BitPath CAST.ERA, ORG), (ONE, CARDINAL), (ONE Media, ORG), (BitPath CAST.ERA, ORG), (ONE, CARDINAL), (ONE Media, ORG), (NextGen Broadcast Standard, PRODUCT), (Doosan Corp MarketLine, ORG), (Doosan Corp, ORG), (Daegu Mao Elementary School Expression, PERSON), (Daegu Mao, ORG), (Eastern Environmental Improvement Electric Sector Design Service, ORG), (Daegu Doosan Elementary School, FAC), (nautmark Eastern Environmental Improvement, ORG), (the Rolls Royce Group's, ORG), (Competition Authority of the French Republic, ORG), (Autorité, ORG), (Framatome, ORG), (EDF, ORG), (Industrial Air Compressor Market, ORG), (17.8, MONEY), (Global Market Insights Inc., ORG), ($17.8 Bn, MONEY), (Doosan Bobcat, PERSON), (Cogniac, ORG), (Sustain Asia Daily, ORG), (Japan, GPE), (26, CARDINAL), (CLSA, ORG), (Japan week, DATE), (26 May CLSA, DATE), (Machines From The Manufacturer Doosan, WORK_OF_ART), (Doosan, ORG), (Korea Morning Notes, ORG), (26, CARDINAL), (CLSA, ORG), (Korea, GPE), (26 May, DATE), (Korea, GPE), (CLSA, ORG), (BBB0, PERSON), (Doosan, ORG), (Seoul, GPE), (Korean, NORP), (Warranty Maintenance Services Doosan Dx200a, ORG), (Doosan, PRODUCT), (Dx200a, PRODUCT), (Excavator, PRODUCT), (dhkcebdxcl0001807, PRODUCT)]"
1,002230.SZ,"[(Iflytek Co Ltd, ORG), (Overnight Returns Marktfeld, PERSON), (Intraday, TIME), (Overnight, TIME), (Marktfeld, PERSON), (China, GPE), (China Software, ORG), (Goldman Sachs, ORG), (Iflytek, ORG), (China Knowledge Online Pte. Ltd., ORG), (1, CARDINAL), (Apr, DATE), (China, GPE), (Goldman Sachs, ORG), (China, GPE), (Zhuiyi AI /, PERSON), (NLP, ORG), (75%, PERCENT), (17, CARDINAL), (China Software, ORG), (Goldman Sachs, ORG), (APAC Quantitative & Systematic Strategy Northbound, ORG), (Credit Suisse, ORG), (HK Bourse Results Announcement, ORG), (33, CARDINAL), (HK Bourse, ORG), (Great Wall Motor 33, ORG), (HK Bourse Announcement, ORG), (Chinasoft International, FAC), (9, CARDINAL), (Chinasoft International 9, ORG), (iFLYTEK Co. Ltd., ORG), (MarketLine, PERSON), (NIKKEI Asia Pacific Corporate, ORG), (19 2, CARDINAL), (NIKKEI, ORG), (NIKKEI Asia Pacific Corporate, ORG), (18 2, CARDINAL), (NIKKEI, ORG), (HK Bourse Announcement, ORG), (China Merchants Direct, ORG), (6, CARDINAL), (Iflytek 002230, DATE), (Mar, PERSON), (China Knowledge Online Pte. Ltd., ORG), (Iflytek, ORG), (002230, PRODUCT), (Mar, ORG), (New Decade, EVENT), (AWE2021, EVENT), (Press Release Kingdee International, ORG), (Kingdee International, ORG), (Annual, DATE), (Kingdee International, ORG), (Annual, DATE), (iFlytek, ORG), (Tindall, ORG), (Iflytek Co Ltd Deliver Long Term Returns Sadif Analytics, ORG), (Iflytek Co Ltd, ORG), (Sadif Analytics, ORG), (NZ, GPE), (Australia, GPE), (China, GPE), (NZ Australia, GPE), (AWE, ORG), (Haier Smart Home Brings, PERSON), (Smart Home Solution, ORG), (Haier, ORG), (Haier Smart Home Brings, PERSON), (Smart Home Solution, ORG), (Haier, ORG), (Haier Smart Home Brings, PERSON), (Smart Home Solution, ORG), (Haier, ORG

In [27]:
grouped_df[grouped_df['RIC'] == 'HYFM.OQ']

,RIC,NER_combined
980,HYFM.OQ,"[(Hydrofarm Holdings Group Inc, ORG), (HYFM, ORG), (Q4, DATE), (Q4 Hydrofarm Holdings Group Inc Earnings Call, ORG), (Q4, DATE), (Hydrofarm Holdings Group Inc, ORG), (Hydrofarm Holdings Group Inc., ORG), (HYFM, ORG), (Q4, DATE), (US Penny Stocks, ORG), (January, DATE), (US, GPE), (Zacks Industry Outlook Highlights West Fraser, ORG), (CalMaine Foods Andersons, ORG), (Hydrofarm, PERSON), (Zacks, ORG), (West Fraser Timber, ORG), (CalMaine Foods, ORG), (Hydrofarm, ORG), (4 Agriculture Products Stocks, ORG), (Watch Despite Industry Concerns, ORG), (4, CARDINAL), (Hydrofarm Holdings Group Inc., ORG), (Invest, ORG), (HYFM, ORG), (Hydrofarm Holdings Group, ORG), (3 US Penny, MONEY), (3, CARDINAL), (US, GPE), (30, MONEY), (Hydrofarm, ORG), (FY24, DATE), (Hydrofarm Holdings Group Inc, ORG), (HYFM, ORG), (Q4, DATE), (Q4 Hydrofarm Holdings Group Inc Earnings Call, ORG), (Q4, DATE), (Hydrofarm Holdings Group Inc, ORG), (Hydrofarm Holdings Group Inc., ORG), (HYFM, ORG), (Q4, DATE), (US Penny Stocks, ORG), (January, DATE), (US, GPE), (Zacks Industry Outlook Highlights West Fraser, ORG), (CalMaine Foods Andersons, ORG), (Hydrofarm, PERSON), (Zacks, ORG), (West Fraser Timber, ORG), (CalMaine Foods, ORG), (Hydrofarm, ORG), (4 Agriculture Products Stocks, ORG), (Watch Despite Industry Concerns, ORG), (4, CARDINAL), (Hydrofarm Holdings Group Inc., ORG), (Invest, ORG), (HYFM, ORG), (Hydrofarm Holdings Group, ORG), (3 US Penny, MONEY), (3, CARDINAL), (US, GPE), (30, MONEY), (Hydrofarm, ORG), (FY24, DATE)]"


In [28]:
grouped_df.shape

(2216, 2)

In [29]:
# Save to Excel
df.to_excel("NER.xlsx", index=False)